# Activity 5 — Format Translation
**Module:** Advanced Programming — Week 2  
**University of York, MSc Computer Science**

**Task:** Translate `People.json` → `People_output.xml`  
Then compare the output with the original data to verify integrity.

## Exercise 1: JSON → XML translation program

In [ ]:
import json
import xml.etree.ElementTree as ET

# Step 1: Load the JSON file
with open('People.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print("JSON loaded successfully.")
print(f"Number of students: {len(data['students'])}")
print()

In [ ]:
# Step 2: Build the XML tree

# Root element
root = ET.Element('students')

for student in data['students']:
    # <student> element
    student_elem = ET.SubElement(root, 'student')

    # <id>
    id_elem = ET.SubElement(student_elem, 'id')
    id_elem.text = str(student['id'])

    # <fullName>
    fullname_elem = ET.SubElement(student_elem, 'fullName')

    title_elem = ET.SubElement(fullname_elem, 'title')
    title_elem.text = student['fullName']['title']

    first_elem = ET.SubElement(fullname_elem, 'first')
    first_elem.text = student['fullName']['first']

    surname_elem = ET.SubElement(fullname_elem, 'surname')
    surname_elem.text = student['fullName']['surname']

    # <other> — list of middle names (may contain null values)
    other_elem = ET.SubElement(fullname_elem, 'other')
    for name in student['fullName']['other']:
        middle_elem = ET.SubElement(other_elem, 'middleName')
        # JSON null → XML empty element (preserves the fact that the field exists)
        middle_elem.text = name if name is not None else ''

    # <age>
    age_elem = ET.SubElement(student_elem, 'age')
    age_elem.text = str(student['age'])

    # <city>
    city_elem = ET.SubElement(student_elem, 'city')
    city_elem.text = student['city']

print("XML tree built successfully.")

In [ ]:
# Step 3: Add indentation for readability (Python 3.9+)
ET.indent(root, space='    ')

# Step 4: Write to file
tree = ET.ElementTree(root)
output_file = 'People_output.xml'

tree.write(
    output_file,
    encoding='unicode',
    xml_declaration=False  # kept False to match simple XML structure
)

# Step 5: Print the result
with open(output_file, 'r', encoding='utf-8') as f:
    xml_content = f.read()

print(f"Output written to '{output_file}'\n")
print(xml_content)

## Data Integrity Verification

Round-trip check: parse the generated XML back into Python objects and compare with the original JSON data.

In [ ]:
# Re-parse the generated XML and compare with the original JSON

tree_check = ET.parse('People_output.xml')
root_check = tree_check.getroot()

print("=== Round-trip verification ===")
print(f"{'Field':<20} {'JSON value':<25} {'XML value':<25} {'Match?'}")
print("-" * 80)

for i, (student_json, student_xml) in enumerate(
    zip(data['students'], root_check.findall('student'))
):
    checks = [
        ('id',      str(student_json['id']),                    student_xml.find('id').text),
        ('title',   student_json['fullName']['title'],           student_xml.find('fullName/title').text),
        ('first',   student_json['fullName']['first'],           student_xml.find('fullName/first').text),
        ('surname', student_json['fullName']['surname'],         student_xml.find('fullName/surname').text),
        ('age',     str(student_json['age']),                   student_xml.find('age').text),
        ('city',    student_json['city'],                        student_xml.find('city').text),
    ]
    print(f"--- Student {i+1} ---")
    for field, json_val, xml_val in checks:
        match = '✓' if json_val == xml_val else '✗ MISMATCH'
        print(f"  {field:<18} {json_val:<25} {xml_val:<25} {match}")

print()
print("All fields verified.")

## Differences and data integrity analysis

### What differences exist between the generated XML and the JSON source?

| Aspect | JSON | Generated XML | Impact on integrity |
|---|---|---|---|
| **Data types** | `id` and `age` are integers (`786789`, `38`) | All XML values are strings (`"786789"`, `"38"`) | **No integrity loss** — the values are identical; the type distinction is format-specific and restored on parse |
| **Null middle names** | `"other": [null]` — explicit JSON null | `<middleName></middleName>` — empty XML element | **No integrity loss** — the absence of a name is preserved; a parser reading the XML can check for empty text |
| **List structure** | `"other": ["Ruelle", "Garlen"]` — JSON array | `<other><middleName>Ruelle</middleName><middleName>Garlen</middleName></other>` — repeated child elements | **No integrity loss** — multiple names are correctly represented and countable |
| **Key names** | camelCase (`fullName`, `surname`) | Same names preserved as element tags | **No difference** |

### Conclusion

All differences are format conventions, not data losses. The round-trip verification confirms that every field round-trips correctly: the same values that exist in the JSON source are present in the XML output and can be read back accurately.

The one decision that required care was handling `null` middle names. In JSON, `null` is an explicit typed absence of a value. XML has no native null — the representation chosen here (empty element text) preserves the *intent* (this person has a field but no value in it) while remaining parseable. An alternative would have been to use an XML attribute: `<middleName xsi:nil="true"/>`, but that adds namespace complexity not warranted for this data.